## Data Intake (Reddit Scraping)

In [1]:
import praw
import time
from datetime import datetime
import re

In [2]:
r = praw.Reddit("watchdog")

In [4]:
cbbbot = r.redditor("cbbBot")

In [15]:
datetime.fromtimestamp(cbbbot.created_utc).strftime("%Y-%m-%d %H:%M:%S")

'2016-11-12 15:44:04'

In [3]:
cbb = r.subreddit("CollegeBasketball")

In [4]:
game_thread_indices = list(cbb.search('"nationally televised games"', sort="new", limit=50))

In [ ]:
from thread_scraper import scrape_index_thread

for index in game_thread_indices:
    scrape_index_thread(index)

FINAL | [ESPN2](#l/espn2) | 125 | Liberty | George Mason | 96 | [Thread](https://www.reddit.com/r/CollegeBasketball/comments/1rwj9ef/game_thread_liberty_george_mason_0600_pm_et/) | [Thread](https://www.reddit.com/r/CollegeBasketball/comments/1rwo72w/post_game_thread_liberty_defeats_george_mason_7771/)
FINAL | [truTV](#l/trutv) | 185 | UMBC | Howard | 207 | [Thread](https://www.reddit.com/r/CollegeBasketball/comments/1rwkcvj/game_thread_16_umbc_16_howard_0640_pm_et/) | [Thread](https://www.reddit.com/r/CollegeBasketball/comments/1rwpfn6/post_game_thread_16_howard_defeats_16_umbc_8683/)
FINAL | [ESPNU](#l/espnu) | 98 | Wyoming | Wichita State | 82 | [Thread](https://www.reddit.com/r/CollegeBasketball/comments/1rwkwac/game_thread_wyoming_wichita_state_0700_pm_et/) | [Thread](https://www.reddit.com/r/CollegeBasketball/comments/1rwpqs1/post_game_thread_wichita_state_defeats_wyoming/)
FINAL | [ESPN2](#l/espn2) | 114 | Davidson | Oklahoma State | 66 | [Thread](https://www.reddit.com/r/College

AttributeError: 'NoneType' object has no attribute 'name'

In [6]:
(time.time() - game_threads[-1].created_utc) / 86400

NameError: name 'game_threads' is not defined

In [7]:
len(game_thread_indices)

50

In [5]:
with open("thread_dump.txt", "w") as f:
    f.write(game_thread_indices[1].selftext)

In [5]:
from data_structures import *
from utils import parse_index_thread

# def parse_index_thread(body_text):
#     games = []

#     table_lines = body_text.split("\n")
#     for line in table_lines:
#         # lines containing game threads begin either with 'FINAL' or a clock time
#         if re.match("[0-9]|F", line[:1]):

#             # character sequence in between the two backslashes will be the post ID
#             game_thread_id = re.search("/[a-z,0-9]+/game", line).group()[1:-5]
#             post_thread_id = re.search("/[a-z,0-9]+/post", line).group()[1:-5]

#             # matches the formatted table columns of KP | Away | Home | KP
#             teams_raw = re.search("[0-9]+( \| .+){2}\| [0-9]+", line).group()
#             teams_list = re.sub(r'#?[0-9]+', "", teams_raw).split(" | ")[1:-1]

#             home_team = teams_list[1].strip()
#             away_team = teams_list[0].strip()
#             games.append(Game(home_team, away_team, None, None, game_thread_id, post_thread_id, None))
    
#     return games

game_tuples = parse_index_thread(game_thread_indices[1].selftext)

In [6]:
game_tuples[0]

Game(home='Arkansas', away='Vanderbilt', home_score=None, away_score=None, game_thread='1ruhrhg', post_thread='1rumvjt', when_played=None)

In [7]:
experiment_thread = r.submission(game_tuples[0].game_thread)
experiment_thread.title

'[Game Thread] #20 Vanderbilt @ #18 Arkansas (01:00 PM ET)'

In [18]:
comment_list = list(r.submission(game_tuples[3].game_thread).comments.list())

In [3]:
from utils import match_team_with_conference

with open("teams.txt", "r") as f:
    teams = [line.strip() for line in f.readlines()]

for team in teams:
    print(match_team_with_conference(team))

SEC
SEC
Big 12
Big 12
ACC
ACC
Big Ten
Big Ten
A10
A10
Big Sky
Big Sky
Big 12
Big 12
SLC
SLC
Big Sky
Big Sky
CUSA
CUSA
MAAC
MAAC
WCC
WCC
SWAC
SWAC
Big Ten
Big Ten
ACC
ACC
America East
America East
Big 12
Big 12
Horizon
Horizon
Northeast
Northeast
SLC
SLC
CAA
CAA
CUSA
CUSA
America East
America East
Big Ten
ACC
ACC
Big 12
Big 12
ACC
SWAC
SWAC
Big 12
WCC
Big Sky
Horizon
WCC
SLC
SWAC
CAA
Big Sky
Horizon
Southern
Southern
Sun Belt
Sun Belt
SLC
CAA
SWAC
WCC
Big Sky
Summit
Summit
MAAC
SLC
Sun Belt
WCC
CAA
Big Sky
Southern
MAAC
SLC
Sun Belt
CAA
Big Ten
Big Ten
Big Ten
Southern
AAC
AAC
Horizon
AAC
AAC
AAC
AAC
Big Ten
CAA
AAC
AAC
AAC
AAC
ASUN
ASUN
Patriot League
Patriot League
Patriot League
Patriot League
Big Ten
CAA
Big South
Big South
MVC
MVC
Big West
Big West
Big Ten
Mountain West
Mountain West
Big Sky
CAA
WCC
Big 12
MAAC
Summit
Big 12
Big Ten
Big Ten
Big West
Big West
Sun Belt
Big East
Big East
OVC
OVC
WAC
WAC
Big Ten
WCC
Mountain West
Mountain West
WAC
WAC
SEC
SEC
SEC
SEC
Big West
Big West


In [ ]:
import yaml

def match_flair_with_team(flair, all_teams):
    if flair is None:
        return
    
    team_codes = re.findall(r":([a-z]+):", flair)
    
    for code in team_codes:
        with open("flairs.yaml", "r") as f:
            flair_mapping = yaml.load(f, Loader=yaml.SafeLoader)
        if code in flair_mapping.keys():
            return flair_mapping[code]
        else:
            matches = []
            for team in all_teams:
                if re.match(code, team.replace(" ", "").lower()):
                    matches.append(team)

            
            if len(matches) == 0:
                manual_match = input(f"Couldn't match '{code}' to a team name. \nEnter team or leave blank for None: ")
                flair_mapping[code] = manual_match if len(manual_match) > 0 else None
            elif len(matches) == 1:
                flair_mapping[code] = matches[0]
            else:
                for match in matches:
                    if team.replace(" ", "").lower() == code:
                        flair_mapping[code] == match
                
                if code not in flair_mapping.keys():
                    for i, match in enumerate(matches):
                        print(f"{i+1}. {match}")
                    selection = input(f"Which team (1-{len(matches)}) does the flair '{code}' refer to? ")
                    flair_mapping[code] = matches[int(selection)-1]

            with open("flairs.yaml", "w") as f:
                yaml.dump(flair_mapping, f)

    return [flair_mapping[code] for code in team_codes]

for comment in comment_list:
    print(match_flair_with_team(comment.author_flair_text))

:dayton: :ohiostate: Dayton Flyers • Ohio State Buckeyes
Dayton
:coloradostate: :gonzaga: Colorado State Rams • Gonzaga Bulldogs
Colorado State
:saintlouis: Saint Louis Billikens
['Saint Louis']
:saintlouis: Saint Louis Billikens
Saint Louis
:virginia: :georgetown: Virginia Cavaliers • Georgetown Hoyas
Virginia
:dayton: Dayton Flyers
Dayton
:dayton: :michiganstate: Dayton Flyers • Michigan State Spartans
Dayton
:pittsburgh: Pittsburgh Panthers
['Pittsburgh']
None
:illinois: Illinois Fighting Illini
Illinois
:dayton: Dayton Flyers
Dayton
:dayton: Dayton Flyers
Dayton
:clemson: :dayton: Clemson Tigers • Dayton Flyers
Clemson
:arizona: :usc: Arizona Wildcats • USC Trojans
Arizona
:dayton: Dayton Flyers
Dayton
:utah: Utah Utes
Utah
None
:vcu: VCU Rams
VCU
:vcu: VCU Rams
VCU
:smu: SMU Mustangs
['SMU']
:vcu: VCU Rams
VCU
:minnesota: Minnesota Golden Gophers
Minnesota
:vcu: :missouri: VCU Rams • Missouri Tigers
VCU
:vcu: VCU Rams
VCU
None
None
:dayton: :villanova: Dayton Flyers • Villanova Wi

In [34]:
with open("thread_dump.txt", "w") as f:
    f.write(r.submission(game_tuples[0].game_thread).selftext)

In [8]:
from utils import parse_game_thread

parse_game_thread(experiment_thread.selftext)

('2026-03-15 13:00:00', ('26', '8', '26', '8'), ('75', '86'))

In [17]:
from utils import extract_team_names

teams = extract_team_names([thread.title if thread.author.name == "cbbBot" else "" for thread in game_threads])

In [3]:
len(teams)

313

In [8]:
from utils import match_flair_with_team

match_flair_with_team("Michigan Wolverines", teams)

'Michigan'

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("unsloth/DeepSeek-R1-Distill-Llama-8B-GGUF", 
                                             gguf_file="DeepSeek-R1-Distill-Llama-8B-BF16.gguf",
                                             torch_dtype=torch.float16
                                             ).to('cuda:1')

/home/max/anaconda3/envs/reddit/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Converting and de-quantizing GGUF tensors...: 100%|██████████| 292/292 [00:48<00:00,  5.99it/s]


In [3]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/DeepSeek-R1-Distill-Llama-8B-GGUF", 
                                          gguf_file="DeepSeek-R1-Distill-Llama-8B-BF16.gguf", 
                                          torch_dtype=torch.float16)

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


In [ ]:
from db import get_table, engine
from sqlalchemy import create_engine, MetaData, Table, select, update, exists, func
from data_structures import *
from typing import List
from utils import user_affiliation_context

def query_team_related_comments(team_name: str, min_length: int = None, limit: int = None) -> List[Comment]:
    comments = get_table("comments")
    games = get_table("games")
    users = get_table("users")

    stmt = (
        select(
            comments,
            games.c.home.label("home"),
            games.c.away.label("away"),
            users.c.flair_1.label("flair_1"),
            users.c.flair_2.label("flair_2")
        )
        .join(games, ((games.c.game_thread == comments.c.post_id) | (games.c.post_thread == comments.c.post_id)))
        .join(users, comments.c.author == users.c.username)
        .where((games.c.home == team_name) | (games.c.away == team_name))
        .where(func.char_length(comments.c.body) >= min_length if min_length is not None else True)
        .limit(limit)
    )

    with engine.connect() as conn:
        start_time = time.time()
        result = conn.execute(stmt)
        end_time = time.time()
        print(f"Query executed in {end_time - start_time} seconds")
        return [f"{row.author} ({user_affiliation_context(row.home, row.away, user_flair_1=row.flair_1, user_flair_2=row.flair_2)}): {row.body}" for row in result]

In [30]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

client = chromadb.Client()
embedding_function = SentenceTransformerEmbeddingFunction(
    model_name="gtr-t5-xl",
    device="cuda:0"
)
collection = client.get_or_create_collection("query_commments", embedding_function=embedding_function)
comments = query_team_related_comments("Virginia", min_length=50, limit=None)
print(len(comments))
collection.add(
    documents=comments,
    ids=[f"comment_{i}" for i in range(len(comments))]
)

2026-04-08 10:27:54,476 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-08 10:27:54,477 INFO sqlalchemy.engine.Engine SELECT comments.id, comments.author, comments.post_id, comments.parent_id, comments.body, comments.upvotes, comments.created_at, games.home AS home, games.away AS away 
FROM comments JOIN games ON games.game_thread = comments.post_id OR games.post_thread = comments.post_id JOIN users ON comments.author = users.username 
WHERE (games.home = %(home_1)s::VARCHAR OR games.away = %(away_1)s::VARCHAR) AND char_length(comments.body) >= %(char_length_1)s::INTEGER
2026-04-08 10:27:54,478 INFO sqlalchemy.engine.Engine [generated in 0.00239s] {'home_1': 'Virginia', 'away_1': 'Virginia', 'char_length_1': 50}
Query executed in 0.07119441032409668 seconds
2026-04-08 10:27:54,546 INFO sqlalchemy.engine.Engine ROLLBACK


AttributeError: Could not locate column in row for column 'flair_1'

In [14]:
prompt_template = """Hi, please give a thorough answer to the following question. Use the provided context below.
In case you can't find enough information, just respond "I could not find enough relevant comments to answer that."

User question: {}

Context:
{}
"""
  
user_question = "What are Duke's offensive strengths and weaknesses?"
results = collection.query(query_texts=[user_question], n_results=10)
context = "\n".join(
    [f"{i+1}. {passage}" for i, passage in enumerate(results["documents"][0])]
)
prompt = f"{prompt_template.format(user_question, context)}"

In [15]:
print(prompt)

Hi, please give a thorough answer to the following question. Use the provided context below.
In case you can't find enough information, just respond "I could not find enough relevant comments to answer that."

User question: What are Duke's offensive strengths and weaknesses?

Context:
1. likeabosstroll: We didn’t play to their weakness and Duke had a lights out night. Our paint defense is really good, but Duke shot like 60% from the three in the first half, and our centers lost their cool. FSU did what we do but with more effective three point blockage. 
2. evitabilities: Picked a bad game to play like this. Also dukes still really good on defende
3. cowmookazee: Duke defense is no joke, even missing two starters. They hammered TDR, smart on their part.
4. Mobile-Tangelo: I’m saying Duke ran themselves into the ground with their short bench
5. Cr0matose: Alright its time for the defense to just lock in.

-Me every Duke thread
6. StoopSign: Well timed. Duke looked like they were only g

In [31]:
messages = [
    {"role": "user", "content": prompt},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=1000)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Okay, so I need to figure out what people are saying about Virginia's defense in their recent games. Let me go through the context provided and extract the relevant comments.

First, I'll scan through each comment to see if anyone mentions Virginia's defense positively or negatively.

1. Ears_2_Hear: Talks about Virginia playing a great game but doesn't specify defense.
2. Lee-Key-Bottoms: Mentions that Virginia is a team they've historically done well against, but doesn't talk about defense.
3. Wematanye_92: Asks why Virginia isn't talked about, no defense here.
4. Newlifter10: Scared of facing Virginia, but no defense mention.
5. stormstopper: Impressed with Virginia's defense, mentions they were physical, got blocks, and forced turnovers. This is a positive note.
6. Larrybirdlover: Compares Virginia's offense to NC State's, saying Virginia's is more team-oriented. No defense here.
7. 954gator: Mentions Virginia needs to be careful about fouls, implying they might be too aggressive o

In [3]:
# !pip install llama-cpp-python

from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="unsloth/DeepSeek-R1-Distill-Llama-8B-GGUF",
	filename="DeepSeek-R1-Distill-Llama-8B-BF16.gguf",
    n_gpu_layers=-1,
    n_batch=512,
)


llama_model_loader: loaded meta data with 39 key-value pairs and 292 tensors from /home/max/.cache/huggingface/hub/models--unsloth--DeepSeek-R1-Distill-Llama-8B-GGUF/snapshots/615f8936e16dfde29dcc00be71145d4d5ce8ed53/./DeepSeek-R1-Distill-Llama-8B-BF16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Deepseek-R1-Distill-Llama-8B
llama_model_loader: - kv   3:                           general.basename str              = Deepseek-R1-Distill-Llama-8B
llama_model_loader: - kv   4:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   5:                         general.size_label str   

In [2]:
llm.create_chat_completion(
	messages = [
		{
			"role": "user",
			"content": "What is the capital of France?"
		}
	]
)

KeyboardInterrupt: 